# Week 7: Content Action Playbook

This notebook transforms the Week 5 model output into a ranked, actionable content playbook.

The goal is not to automate content decisions. The goal is to make human review faster and more focused.

The playbook will include:
- Ranked actions with reason codes (why this page matters)
- Archetype → action mapping (what kind of pages get what treatment)
- Clear intended use (when to use this list, when not to)
- Known limits (what can go wrong)
- Human-review rules (when to override the ranking)
- Monitoring and retrain triggers (how to know if the model drifts)
- What NOT to automate (the guardrails)

The output goes to `work/outputs/` for the paper next week.

## 0) Setup

Load the model output, baseline, and feature data from Week 5.

In [ ]:
from pathlib import Path
import subprocess
import sys
import json

import numpy as np
import pandas as pd

# Repo setup
root = Path.cwd()
if not (root / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists():
    for candidate in [Path.cwd(), Path.cwd().parent]:
        if (candidate / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists():
            root = candidate
            break

sys.path.insert(0, str(root / 'scripts'))
sys.path.insert(0, str(root))

# Ensure output directories exist
output_dir = root / 'work' / 'outputs'
figure_dir = root / 'work' / 'figures'
output_dir.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)

print(f'Repo root: {root}')
print(f'Output dir: {output_dir}')
print(f'Figure dir: {figure_dir}')

In [ ]:
# Prepare features if needed
feature_path = root / 'data' / 'processed' / 'refresh_feature_vector.csv'
baseline_path = root / 'data' / 'processed' / 'baseline_refresh_queue.csv'
model_results_path = root / 'outputs' / 'model_results.json'

if not feature_path.exists():
    subprocess.run(['python', str(root / 'scripts' / '01_prepare_features.py')], cwd=root, check=True)

if not baseline_path.exists():
    subprocess.run(['python', str(root / 'scripts' / '02_baseline_score.py')], cwd=root, check=True)

if not model_results_path.exists():
    subprocess.run(['python', str(root / 'scripts' / '03_train_model.py')], cwd=root, check=True)

# Load data
frame = pd.read_csv(feature_path)
baseline_queue = pd.read_csv(baseline_path)

# Try to load model results
try:
    with open(model_results_path) as f:
        model_results = json.load(f)
    print(f'Loaded model results from {model_results_path}')
except FileNotFoundError:
    print(f'Model results not found; will use baseline metrics')
    model_results = {}

# Load predictions if available
prediction_path = root / 'data' / 'processed' / 'model_predictions.csv'
if prediction_path.exists():
    predictions = pd.read_csv(prediction_path)
    print(f'Loaded {len(predictions):,} predictions')
else:
    print('No prediction CSV found; will use baseline scores')
    predictions = None

print(f'Frame shape: {frame.shape}')
print(f'Baseline queue shape: {baseline_queue.shape}')

## 1) Ranked Actions + Reason Codes

Transform the model output and baseline into an actionable ranked queue.

Each row should include:
- Content ID (pseudonymized)
- Model score or ranking
- Reason code (why this page is in the queue)
- Recommended action (refresh, expand, rewrite metadata, monitor, etc.)
- Confidence level (high, medium, low based on data volume and signal clarity)
- Key metrics (impressions, position, CTR, engagement for context)

In [ ]:
# Start with baseline queue as the foundation
action_queue = baseline_queue[[
    'content_id', 'client_id', 'baseline_refresh_score', 'baseline_rank',
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'avg_position', 'ctr',
    'engagement_rate', 'scroll_rate', 'content_age_days', 'days_since_last_update',
    'reason_codes', 'suggested_action_baseline'
]].copy()

# If model predictions exist, blend them with baseline
if predictions is not None:
    action_queue = action_queue.merge(
        predictions[['content_id', 'best_model_name', 'best_model_probability']],
        on='content_id',
        how='left'
    )
    # Create a blended score: 70% model, 30% baseline
    action_queue['model_probability'] = action_queue['best_model_probability'].fillna(0.5)
    action_queue['blended_score'] = (
        0.7 * action_queue['model_probability'] +
        0.3 * (action_queue['baseline_refresh_score'] / 100)
    )
else:
    action_queue['model_probability'] = np.nan
    action_queue['blended_score'] = action_queue['baseline_refresh_score'] / 100

# Assign action labels based on score and reason codes
def assign_action_label(row):
    reasons = str(row['reason_codes']).lower().split('|')
    score = row['blended_score']
    
    if score >= 0.75:
        if 'thin_visible_page' in reasons:
            return 'expand_and_refresh'
        elif 'low_ctr_visible_page' in reasons or 'low_engagement_visible_page' in reasons:
            return 'review_content_quality'
        else:
            return 'prioritize_refresh'
    elif score >= 0.50:
        if 'stale_visible_page' in reasons:
            return 'refresh_for_freshness'
        else:
            return 'review_strategy'
    elif score >= 0.25:
        return 'monitor'
    else:
        return 'no_action'

action_queue['action_label'] = action_queue.apply(assign_action_label, axis=1)

# Assign confidence based on data availability
def assign_confidence(row):
    if row['impressions_90d'] >= 500 and row['sessions_90d'] >= 30:
        return 'high'
    elif row['impressions_90d'] >= 100 and row['sessions_90d'] >= 5:
        return 'medium'
    else:
        return 'low'

action_queue['confidence'] = action_queue.apply(assign_confidence, axis=1)

# Sort by blended score (highest first)
action_queue = action_queue.sort_values('blended_score', ascending=False).reset_index(drop=True)
action_queue['action_rank'] = action_queue.index + 1

print(f'Action queue built with {len(action_queue):,} rows')
print(f'\nTop 20 actions:')
display_cols = ['action_rank', 'content_id', 'action_label', 'blended_score', 'confidence', 'impressions_90d', 'ctr', 'avg_position']
print(action_queue[display_cols].head(20).to_string(index=False))

In [ ]:
# Show distribution of actions
action_counts = action_queue['action_label'].value_counts()
print('Action distribution:')
print(action_counts.to_string())
print('')
print('Confidence distribution:')
confidence_counts = action_queue['confidence'].value_counts()
print(confidence_counts.to_string())

## 2) Intended Use and Limits

The playbook is only useful if humans understand exactly what it is and what it is not.

### Intended Use

**What this playbook is for:**

1. **Prioritizing human review**: Use this ranked list to decide which pages a content reviewer should examine first.
2. **Focused action suggestions**: For each page, the reason codes explain why it appears in the queue (stale, low-CTR, declining, etc.).
3. **Relative ranking**: Comparing two pages on this list tells you which one has clearer signals for review.
4. **Light decision support**: Use the action labels as starting suggestions, not as automatic decisions.

**Who should use it:**

- Content strategists deciding which pages to prioritize
- SEO managers reviewing quarterly content audits
- Teams with limited review capacity who want to focus on the highest-impact candidates first

**When to use it:**

- Quarterly or monthly content planning
- After significant traffic or ranking changes
- As an input to larger content strategy discussions

### Known Limits

1. **Not a guarantee**: Pages ranked high on this list are more likely to show the signals (decline, low engagement, freshness opportunity) that the model learned. It is not a guarantee they will recover if refreshed.
2. **Snapshot in time**: The scores are based on a 90-day window of historical data. Recent changes (last 3 days) are not included.
3. **Aggregate signals only**: The model does not see raw query text, content titles, or user comments—only measurements like position, impressions, CTR, and engagement.
4. **Confounded by position**: Pages in low-rank positions will naturally have lower CTR, even if the content is good. The model captures this pattern but cannot distinguish low-CTR-because-bad-ranking from low-CTR-because-bad-content without more analysis.
5. **Not personalized**: The same action suggestion applies to all pages in an archetype, even though individual pages may have unique contexts (e.g., seasonal demand, recent updates, known competitors).
6. **Low-volume pages add noise**: Pages with fewer than 100 impressions in 90 days have noisier signals; confidence levels account for this.

### When NOT to Use This Playbook

- **Do not automate refresh decisions** based on the score alone. Always review the page first.
- **Do not assume declining pages will recover** without understanding why they declined (algorithm update, user behavior, seasonality, consolidation).
- **Do not ignore pages not in the queue**. This is a priority list, not an audit of all pages.
- **Do not treat high confidence as infallible**. High-volume pages have clearer signals, but can still be outliers.
- **Do not skip human judgment**. If a page looks important for other reasons (brand, SEO foundation, user feedback), review it regardless of rank.

## 3) Human-Review Rules + The No-Go List

Define the guardrails: when humans should override the model, and what should never be automated.

### Human-Review Rules

**Always review, even if ranked low:**

1. Pages with recent major changes (refreshed in the last 14 days): The signals may not have time to reflect the change yet.
2. Brand-critical pages or foundational content: Reviews, case studies, core service pages, homepage, about page.
3. Pages with known seasonal demand: Holiday guides, event pages, topic-specific content with strong seasonality.
4. Pages that have shown recent ranking volatility (±20 position swings in recent weeks).
5. Pages where you know there are pending SEO improvements (pending title/meta updates, new backlinks expected, technical fixes in progress).

**Adjust action priority based on:**

1. **Known competitor activity**: If a competitor has recently published on the same topic, lower-ranked pages may be worth refreshing sooner.
2. **Traffic opportunity size**: A page with 10,000 impressions and 0.5% CTR has more upside than a page with 100 impressions and 0.2% CTR, even if both are ranked high.
3. **Refresh complexity**: Pages that are easier to update (simple technical rewrites) may be worth higher priority than pages that require complete rewrites.
4. **Team capacity and lead time**: If a full refresh takes 4 weeks, start with pages where the ROI justifies that timeline.

### The No-Go List: What NOT to Automate

**Never automate:**

1. **Content deletion or archiving** based on this model alone. Traffic loss could indicate consolidation elsewhere on the site, and deleting could cause 404s or cannibalization.
2. **Wholesale title/meta rewrites** without human review. Metadata changes should be guided by keyword research, competitor analysis, and brand voice, not just declining visibility.
3. **Merge or consolidation decisions** based on low rank. Related pages might serve different user intents or capture different search patterns.
4. **Content licensing or republishing decisions** based on engagement scores. External factors matter.
5. **Changes to internal link structure** based on this ranking alone. Link strategy should account for information architecture, crawl efficiency, and user flow.

**What should remain manual and intentional:**

- Strategy decisions (which topics to own, which to exit)
- Tone and brand voice changes
- Major rewrites or repositioning
- Decisions to stop covering a topic
- Technical SEO changes that affect multiple pages

**What is safe to assist (with human sign-off):**

- Generating refresh suggestions ("add recent data, update examples")
- Flagging outdated claims or broken links for manual review
- Recommending structural improvements (more headings, better formatting)
- Batching pages for the review queue

In [ ]:
# Identify pages that should be reviewed manually even if ranked lower
no_go_flags = []

# Flag brand-critical pages (low volume but high brand importance would show in client patterns)
# For now, just flag very new or very old pages
action_queue['review_override_reason'] = ''

action_queue.loc[action_queue['content_age_days'] < 14, 'review_override_reason'] = 'Recently created; signals may not be stable yet'
action_queue.loc[action_queue['content_age_days'] > 730, 'review_override_reason'] = 'Very old; may need foundational review'
action_queue.loc[action_queue['impressions_90d'] < 50, 'review_override_reason'] = 'Very low volume; treat signal as directional only'

override_count = (action_queue['review_override_reason'] != '').sum()
print(f'Pages with manual review override flags: {override_count}')
print('')
print('Override reasons:')
print(action_queue['review_override_reason'].value_counts().to_string())

## 4) Monitoring and Retrain Triggers

Define what success looks like and when the model needs to be retrained.

### How to Monitor the Playbook

**Metric 1: Refresh Action Compliance**
- Track how many pages recommended for refresh are actually refreshed (closure rate).
- Target: 40–60% of high-confidence refresh recommendations are acted on within 90 days.
- If <20%, the list may not be credible or actionable; investigate why.

**Metric 2: Outcome After Action**
- For pages that were refreshed, measure the outcome 30–60 days after:
  - Did impressions recover, stay flat, or decline further?
  - Did CTR improve, stay flat, or worsen?
  - Did engagement change?
- Track a cohort of refreshed pages from the high-confidence, high-score bucket and compare to a random sample of other refreshed pages.
- Win rate: pages refreshed from this list should show at least 40–50% positive outcomes (modest recovery or stabilization).

**Metric 3: Model Drift**
- Re-run the model monthly and compare the top-50 recommendations to last month's top-50.
- Stability: 60–70% overlap is healthy. >90% overlap suggests the model is stale; <40% overlap suggests instability or a real shift in the data.
- If drift is detected, check for:
  - Algorithm updates (Google, Bing).
  - Traffic seasonality.
  - Site-wide changes or new competitors.

### Retrain Triggers

**Retrain the model if:**

1. **Quarterly cadence**: Retrain at least every 90 days with fresh data. The world changes; the model should too.
2. **Major site event**: After a site redesign, domain migration, or large content push, retrain within 2 weeks.
3. **Significant outcome divergence**: If >50% of high-score refreshed pages show negative outcomes (continued decline), retrain and audit the features.
4. **Feature data quality degradation**: If metrics become systematically unavailable or unreliable (e.g., GSC data stops, analytics changes), retrain with updated definitions.
5. **Ranked list instability**: If the top-50 changes by >40% month-over-month without a clear trigger, investigate and retrain.

### What to Track in a Monitoring Log

Create a simple CSV or sheet to log:
- Deployment date (when this playbook was distributed)
- Top-50 recommendations (save as a snapshot)
- Number of pages reviewed by month
- Number of pages refreshed from recommendations
- Outcome distribution (improved, flat, declined)
- Major events (algorithm updates, site changes, data quality issues)
- Next retrain date

This log becomes the evidence trail for future iterations.

In [ ]:
# Export a monitoring template
monitoring_log = pd.DataFrame({
    'date': [pd.Timestamp.now().strftime('%Y-%m-%d')],
    'playbook_version': ['w07_v1'],
    'top_50_recommendations': [action_queue.head(50)['content_id'].tolist()],
    'total_actions_recommended': [len(action_queue)],
    'high_confidence_count': [(action_queue['confidence'] == 'high').sum()],
    'medium_confidence_count': [(action_queue['confidence'] == 'medium').sum()],
    'low_confidence_count': [(action_queue['confidence'] == 'low').sum()],
    'pages_reviewed_this_month': [0],
    'pages_refreshed_from_recommendations': [0],
    'positive_outcomes_count': [0],
    'flat_outcomes_count': [0],
    'negative_outcomes_count': [0],
    'major_events': ['Initial deployment'],
    'next_retrain_date': [(pd.Timestamp.now() + pd.Timedelta(days=90)).strftime('%Y-%m-%d')],
    'notes': ['Baseline monitoring log for w07 playbook']
})

print('Monitoring log template:')
print(monitoring_log.to_string(index=False))

## 5) Exports for the Paper

Generate the CSV and JSON files that will be referenced in next week's research paper.

In [ ]:
# Export the main action queue (this is the core deliverable)
queue_export_cols = [
    'action_rank', 'content_id', 'client_id',
    'blended_score', 'baseline_refresh_score', 'model_probability',
    'action_label', 'confidence', 'review_override_reason',
    'reason_codes', 'suggested_action_baseline',
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr', 'avg_position',
    'engagement_rate', 'scroll_rate', 'content_age_days', 'days_since_last_update'
]

queue_export = action_queue[queue_export_cols].head(500).copy()  # Export top 500 for review
queue_path = output_dir / 'action_playbook_queue.csv'
queue_export.to_csv(queue_path, index=False)
print(f'Exported action queue to: {queue_path}')
print(f'  Rows: {len(queue_export):,}')
print(f'  Columns: {len(queue_export.columns)}')

In [ ]:
# Export playbook metadata and metrics
playbook_metadata = {
    'created_at': pd.Timestamp.now().isoformat(),
    'playbook_version': 'w07_v1',
    'total_pages_in_queue': int(len(action_queue)),
    'top_50_actions': action_queue.head(50)[['action_rank', 'content_id', 'action_label', 'blended_score']].to_dict('records'),
    'action_distribution': action_queue['action_label'].value_counts().to_dict(),
    'confidence_distribution': action_queue['confidence'].value_counts().to_dict(),
    'metrics': {
        'high_confidence_count': int((action_queue['confidence'] == 'high').sum()),
        'medium_confidence_count': int((action_queue['confidence'] == 'medium').sum()),
        'low_confidence_count': int((action_queue['confidence'] == 'low').sum()),
        'average_blended_score': float(action_queue['blended_score'].mean()),
        'median_impressions_90d': int(action_queue['impressions_90d'].median()),
        'median_ctr': float(action_queue['ctr'].median()),
        'median_engagement_rate': float(action_queue['engagement_rate'].median()),
    },
    'intended_use': [
        'Prioritize human content review',
        'Suggest refresh/expand/rewrite actions based on signals',
        'Support quarterly content planning',
        'Enable focused review of high-impact pages'
    ],
    'known_limits': [
        'Snapshot of 90-day historical data (last 3 days not included)',
        'Signals are aggregate, not causal',
        'Low-volume pages (<100 impressions) have noisy signals',
        'Position is a strong confounder for CTR',
        'No guarantee of recovery after refresh'
    ],
    'no_go_list': [
        'Do not automate content deletion based on score',
        'Do not assume declining pages will recover without investigation',
        'Do not skip human judgment on brand-critical pages',
        'Do not treat high confidence as infallible',
        'Do not make strategy decisions without understanding context'
    ],
    'monitoring_targets': {
        'refresh_compliance_rate': '40-60% of high-confidence refreshes acted on',
        'positive_outcome_rate': '40-50% of refreshed pages show improvement',
        'model_stability': '60-70% overlap in top-50 month-over-month',
        'retrain_cadence': 'Quarterly or after major site events'
    }
}

metadata_path = output_dir / 'playbook_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(playbook_metadata, f, indent=2, default=str)
print(f'Exported playbook metadata to: {metadata_path}')

In [ ]:
# Export summary statistics for paper
summary_stats = {
    'playbook_summary': {
        'total_pages_ranked': int(len(action_queue)),
        'date_generated': pd.Timestamp.now().strftime('%Y-%m-%d'),
        'data_window': '90-day trailing window',
        'confidence_breakdown': {
            'high': int((action_queue['confidence'] == 'high').sum()),
            'medium': int((action_queue['confidence'] == 'medium').sum()),
            'low': int((action_queue['confidence'] == 'low').sum())
        }
    },
    'action_distribution': action_queue['action_label'].value_counts().to_dict(),
    'top_actions_by_score': action_queue.head(20)[['action_rank', 'content_id', 'blended_score', 'action_label']].to_dict('records'),
    'signal_distribution': {
        'impressions_mean': float(action_queue['impressions_90d'].mean()),
        'impressions_median': float(action_queue['impressions_90d'].median()),
        'ctr_mean': float(action_queue['ctr'].mean()),
        'ctr_median': float(action_queue['ctr'].median()),
        'avg_position_mean': float(action_queue['avg_position'].mean()),
        'avg_position_median': float(action_queue['avg_position'].median())
    }
}

summary_path = output_dir / 'playbook_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary_stats, f, indent=2, default=str)
print(f'Exported summary stats to: {summary_path}')

In [ ]:
# Generate and save a visualization for the paper
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Content Action Playbook Summary', fontsize=16, fontweight='bold')

# Action distribution
ax = axes[0, 0]
action_counts = action_queue['action_label'].value_counts()
action_counts.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Action Distribution')
ax.set_xlabel('Number of Pages')

# Confidence distribution
ax = axes[0, 1]
confidence_order = ['high', 'medium', 'low']
confidence_counts = action_queue['confidence'].value_counts().reindex(confidence_order)
confidence_counts.plot(kind='bar', ax=ax, color=['green', 'orange', 'red'])
ax.set_title('Confidence Distribution')
ax.set_ylabel('Number of Pages')
ax.set_xlabel('Confidence Level')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=0)

# Score distribution
ax = axes[1, 0]
ax.hist(action_queue['blended_score'], bins=30, color='teal', edgecolor='black')
ax.set_title('Blended Score Distribution')
ax.set_xlabel('Score')
ax.set_ylabel('Count')
ax.axvline(action_queue['blended_score'].median(), color='red', linestyle='--', label=f'Median: {action_queue["blended_score"].median():.2f}')
ax.legend()

# Impressions vs Score (scatter)
ax = axes[1, 1]
scatter = ax.scatter(action_queue['impressions_90d'], action_queue['blended_score'], 
                     c=action_queue['ctr'], cmap='viridis', alpha=0.6, s=30)
ax.set_title('Impressions vs. Blended Score (colored by CTR)')
ax.set_xlabel('Impressions (90d, log scale)')
ax.set_ylabel('Blended Score')
ax.set_xscale('log')
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('CTR (%)')

plt.tight_layout()
fig_path = figure_dir / 'playbook_summary.png'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
print(f'Saved playbook summary figure to: {fig_path}')
plt.show()

## 6) Self-Check

Before submitting, confirm all components:

### Self-Check Checklist

- [ ] **Ranked actions**: Top pages are ordered by blended score; each has an action label and reason code.
- [ ] **Confidence levels**: High, medium, low confidence are assigned based on data volume and signal clarity.
- [ ] **Archetype → action mapping**: Pages are grouped by signal pattern (stale, low-CTR, declining) with appropriate actions.
- [ ] **Intended use**: Clearly documented that this is for prioritizing human review, not automating decisions.
- [ ] **Known limits**: Spelled out snapshot nature, aggregate signals, position confounding, low-volume noise.
- [ ] **No-go list**: Explicitly lists what should NOT be automated (content deletion, consolidation, major rewrites without review).
- [ ] **Human-review rules**: Documented override conditions (recent changes, brand-critical, seasonal, volatility).
- [ ] **Monitoring framework**: Defined metrics to track compliance, outcomes, and model drift.
- [ ] **Retrain triggers**: Clear conditions for when the model should be retrained (quarterly, post-major-event, high divergence).
- [ ] **Exports for paper**: 
  - [ ] action_playbook_queue.csv (main ranked list)
  - [ ] playbook_metadata.json (methodology and limits)
  - [ ] playbook_summary.json (summary metrics for paper)
  - [ ] playbook_summary.png (visualization)
- [ ] **Notebook committed**: The notebook is saved to `work/notebooks/w07_action_playbook.ipynb`.
- [ ] **Outputs in work/outputs/**: All CSV and JSON files for paper are in `work/outputs/` (which is .gitignored for data files but the notebook regenerates them).
- [ ] **Figures committed**: Charts and visualizations are saved to `work/figures/` and committed.
- [ ] **Cost/value framing**: Discussed relative impact (high-impression pages have more upside) and team capacity constraints.
- [ ] **Language is practical**: Recommendations are actionable by a content team, not overstated or too technical.

### What's Ready for the Paper

The following files are ready to be cited and used in the Week 8 paper:

1. **`work/outputs/action_playbook_queue.csv`** — The ranked action list for case studies or appendix.
2. **`work/outputs/playbook_metadata.json`** — Methodology, limits, and monitoring plan.
3. **`work/outputs/playbook_summary.json`** — Summary statistics for results section.
4. **`work/figures/playbook_summary.png`** — Distribution and correlation charts.

All of these trace back to the Week 5 model and Week 6 validation, which are documented in their respective notebooks and metric files.

In [ ]:
# Final summary of deliverables
print('=== WEEK 7 PLAYBOOK SUMMARY ===')
print('')
print(f'Total pages ranked: {len(action_queue):,}')
print(f'Top actions (1-5):') 
for i, row in action_queue.head(5).iterrows():
    print(f"  {row['action_rank']}. {row['content_id']}: {row['action_label']} (score: {row['blended_score']:.3f}, confidence: {row['confidence']})")
print('')
print('Confidence breakdown:')
for conf in ['high', 'medium', 'low']:
    count = (action_queue['confidence'] == conf).sum()
    pct = 100 * count / len(action_queue)
    print(f"  {conf.capitalize()}: {count:,} ({pct:.1f}%)")
print('')
print('Exports ready for paper:')
print(f"  ✓ {queue_path.name}")
print(f"  ✓ {metadata_path.name}")
print(f"  ✓ {summary_path.name}")
print(f"  ✓ {fig_path.name}")
print('')
print('Key playbook principles:')
print("  - Prioritize human review, don't automate decisions")
print("  - Understand limits: signals are aggregate, not causal")
print("  - Monitor compliance and outcomes quarterly")
print("  - Retrain every 90 days or after major events")
print("  - Apply human judgment to override model on brand-critical pages")